In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.signal import freqz
from ipywidgets import FloatSlider, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# BACKWARD-DIFFERENCE RC IIR DESIGN
# H(s) = 1 / (RCs + 1)
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# FIXED ANALOG PARAMETERS
# ============================================================

R = 1.0
C = 1.0
RC = R*C

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.ex-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.ex-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.ex-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:15px;
    line-height:1.55;
    margin-bottom:9px;
}

.ex-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:10px 12px;
    margin-bottom:8px;
    font-size:14.5px;
    line-height:1.50;
}

.ex-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:15.5px;
    margin-bottom:6px;
}

.ex-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.ex-col{
    flex:1;
    min-width:0;
}

.widget-label{
    font-size:14px !important;
}

.jupyter-widgets input{
    font-size:13.5px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="ex-root">

<div class="ex-header">
Backward-Difference Design of an RC Low-Pass IIR Filter
</div>

<div class="ex-doc">

<b>Analog prototype.</b>
Consider the first-order RC low-pass filter described by

<br><br>

<div style="text-align:center;font-size:16px;">
<b>H(s) = 1 / (RCs + 1).</b>
</div>

<br>

For this demonstration, the circuit parameters are fixed at
<b>R = 1</b> and <b>C = 1</b>, so that <b>RC = 1</b>.

<br><br>

Using the backward-difference substitution

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>s = (1-z<sup>-1</sup>)/T</b>
</div>

the corresponding digital IIR filter becomes

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>
H(z) =
[T/(RC+T)] /
[1 - RC/(RC+T) z<sup>-1</sup>].
</b>
</div>

The sampling period T determines the location of the digital pole and therefore affects
the frequency and impulse responses of the resulting digital filter.

</div>

</div>
"""))

# ============================================================
# CONTROL
# ============================================================

T_slider = FloatSlider(value=0.10,min=0.02,max=0.50,step=0.01,description='Sampling period T:',continuous_update=True,readout_format='.2f',style={'description_width':'125px'},layout=Layout(width='420px'))

design_title = HTML('<div class="ex-title" style="margin:0;">Design parameter</div>',layout=Layout(width='170px'))

controls = HBox([
    design_title,
    T_slider
],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='9px 12px',margin='0 0 8px 0',align_items='center'))

info = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 8px 0'))

# ============================================================
# ANALOG POLE
# ============================================================

analog_pole = -1.0/RC

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.6))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. ANALOG POLE
# ============================================================

ax1.axhline(0,color='black',linewidth=0.8)
ax1.axvline(0,color='black',linewidth=0.8)
ax1.axvspan(-3,0,alpha=0.05)

analog_pole_plot, = ax1.plot([analog_pole],[0],'rx',markersize=8,markeredgewidth=1.8,label='Analog pole')

ax1.set_xlim(-3,1)
ax1.set_ylim(-1.5,1.5)

ax1.set_title('Analog Pole in the s-Plane')
ax1.set_xlabel(r'$\Re\{s\}$')
ax1.set_ylabel(r'$\Im\{s\}$')

ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),frameon=False)

# ============================================================
# 2. DIGITAL POLE
# ============================================================

theta = np.linspace(0,2*np.pi,1000)

unit_circle_x = np.cos(theta)
unit_circle_y = np.sin(theta)

bd_circle_x = 0.5+0.5*np.cos(theta)
bd_circle_y = 0.5*np.sin(theta)

ax2.axhline(0,color='black',linewidth=0.8)
ax2.axvline(0,color='black',linewidth=0.8)

ax2.plot(unit_circle_x,unit_circle_y,'--',linewidth=1.1,label='Unit circle')
ax2.plot(bd_circle_x,bd_circle_y,linewidth=1.1,label='Backward-difference circle')

digital_pole_plot, = ax2.plot([],[],'rx',markersize=8,markeredgewidth=1.8,label='Digital pole')

ax2.set_xlim(-1.2,1.2)
ax2.set_ylim(-1.2,1.2)
ax2.set_aspect('equal',adjustable='box')

ax2.set_title('Digital Pole in the z-Plane')
ax2.set_xlabel(r'$\Re\{z\}$')
ax2.set_ylabel(r'$\Im\{z\}$')

ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 3. ANALOG AND DIGITAL MAGNITUDE RESPONSES
# ============================================================

analog_magnitude_line, = ax3.plot([],[],'--',linewidth=1.2,label='Analog magnitude response')
digital_magnitude_line, = ax3.plot([],[],color='red',linewidth=1.4,label='Digital magnitude response')

ax3.set_xlim(0,1)
ax3.set_ylim(0,1.10)

ax3.set_title('Analog and Digital Magnitude Responses')
ax3.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax3.set_ylabel(r'$|H|$')

ax3.grid(True,linestyle=':',alpha=0.25)
ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 4. DIGITAL IMPULSE RESPONSE
# ============================================================

impulse_markers, = ax4.plot([],[],'ro',markersize=3.4)

stem_collection = LineCollection([],colors='red',linewidths=0.9)
ax4.add_collection(stem_collection)

zero_line = ax4.axhline(0,color='black',linewidth=0.8)

ax4.set_xlim(-1,50)
ax4.set_ylim(-0.01,0.35)

ax4.set_title('Digital IIR Impulse Response')
ax4.set_xlabel('Sample index $n$')
ax4.set_ylabel('$h[n]$')

ax4.grid(True,linestyle=':',alpha=0.25)

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.30,hspace=0.62)

# ============================================================
# UPDATE
# ============================================================

def update_filter(change=None):

    T = T_slider.value

    # --------------------------------------------------------
    # Digital transfer function
    #
    # H(z) = b0 / (1 - a z^-1)
    # --------------------------------------------------------

    b0 = T/(RC+T)
    a = RC/(RC+T)

    b = np.array([b0])
    den = np.array([1.0,-a])

    # --------------------------------------------------------
    # Digital pole
    # --------------------------------------------------------

    digital_pole = a

    # --------------------------------------------------------
    # Digital frequency response
    # --------------------------------------------------------

    omega,Hd = freqz(b,den,worN=32768)

    fn = omega/np.pi
    digital_magnitude = np.abs(Hd)

    # --------------------------------------------------------
    # Analog magnitude response
    #
    # Physical analog frequency:
    # Omega = omega / T
    # --------------------------------------------------------

    Omega = omega/T
    analog_magnitude = 1.0/np.sqrt(1.0+(Omega*RC)**2)

    # --------------------------------------------------------
    # Digital impulse response
    #
    # h[n] = b0 * a^n u[n]
    # --------------------------------------------------------

    n_impulse = 50
    n = np.arange(n_impulse)

    h = b0*(a**n)

    # --------------------------------------------------------
    # Update digital pole
    # --------------------------------------------------------

    digital_pole_plot.set_data([digital_pole],[0])

    # --------------------------------------------------------
    # Update frequency responses
    # --------------------------------------------------------

    analog_magnitude_line.set_data(fn,analog_magnitude)
    digital_magnitude_line.set_data(fn,digital_magnitude)

    # --------------------------------------------------------
    # Update impulse response
    # --------------------------------------------------------

    impulse_markers.set_data(n,h)

    segments = [np.array([[ni,0],[ni,hi]]) for ni,hi in zip(n,h)]
    stem_collection.set_segments(segments)

    # --------------------------------------------------------
    # Numerical information
    # --------------------------------------------------------

    info.value = f"""
    <div class="ex-root">

    <div class="ex-box">

    <div class="ex-title">Current digital filter</div>

    <div class="ex-cols">

    <div class="ex-col">
    Circuit parameters:<br>
    <b>R = {R:.1f}</b><br>
    <b>C = {C:.1f}</b><br>
    <b>RC = {RC:.1f}</b>
    </div>

    <div class="ex-col">
    Sampling period:<br>
    <b>T = {T:.2f}</b><br><br>
    Analog pole:<br>
    <b>s = {analog_pole:.5f}</b>
    </div>

    <div class="ex-col">
    Digital coefficients:<br>
    <b>b₀ = {b0:.6f}</b><br>
    <b>a = {a:.6f}</b>
    </div>

    <div class="ex-col">
    Digital pole:<br>
    <b>z = {digital_pole:.6f}</b><br>
    <b>|z| = {abs(digital_pole):.6f}</b><br>
    Stability: <b>{"STABLE" if abs(digital_pole)<1 else "UNSTABLE"}</b>
    </div>

    </div>

    </div>

    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# EVENT
# ============================================================

T_slider.observe(update_filter,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)
display(info)
display(fig.canvas)

update_filter()